# 入住时长与提前预订行为分析

**分析目标**：
1. 分析用户入住晚数（stay_length）和提前预订时间（lead_time）的分布特征
2. 探究入住时长、提前预订时间与取消率、ADR、预估收入的关系
3. 对比家庭客户与非家庭客户的行为差异
4. 分析不同季节、不同酒店类型下的入住行为模式

**数据口径**：
- 预订量、取消率基于全量数据
- ADR 与预估收入仅统计 is_canceled == 0 的有效订单
- estimated_revenue = adr × total_nights

> **注意**：`set_plot_style()` 已修复中文字体配置，请 **按顺序运行本 Notebook 中所有生成图表的单元格**，以确保保存的图片中文正常显示。需要运行的图表单元格包括：第 2、3、4、5、6、7、8 步中的绘图 + `save_figure()` 调用。

In [ ]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"项目根目录: {PROJECT_ROOT}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import PROCESSED_DATA_PATH, FIGURE_DIR
from src.analysis import (
    load_cleaned_data,
    add_stay_lead_features,
    calculate_stay_length_summary,
    calculate_lead_time_summary,
    calculate_family_behavior_summary,
    calculate_season_stay_summary,
    calculate_hotel_stay_summary,
)
from src.visualization import set_plot_style, save_figure

set_plot_style()
os.makedirs(FIGURE_DIR, exist_ok=True)

print("导入完成。")

---
## 第 1 步：读取数据并构造行为分析字段

In [ ]:
df = load_cleaned_data(PROCESSED_DATA_PATH)
print(f"原始数据集: {df.shape[0]:,} 行")

df = add_stay_lead_features(df)
print(f"新增字段: total_nights, estimated_revenue, is_family, stay_length_group")
print(f"\n入住晚数分布: min={df['total_nights'].min()}, max={df['total_nights'].max()}, median={df['total_nights'].median():.0f}")
print(f"提前预订天数分布: min={df['lead_time'].min()}, max={df['lead_time'].max()}, median={df['lead_time'].median():.0f}")
print(f"家庭客户占比: {df['is_family'].mean():.1%}")

---
## 第 2 步：入住时长分布分析

In [ ]:
stay_stats = calculate_stay_length_summary(df)
stay_stats

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
colors = sns.color_palette("Set2", len(stay_stats))

bars1 = ax1.bar(stay_stats["stay_length_group"], stay_stats["booking_count"], color=colors)
ax1.set_title("入住时长分布", fontsize=13, fontweight="bold")
ax1.set_xlabel("入住晚数分组")
ax1.set_ylabel("预订量")
for bar, val, pct in zip(bars1, stay_stats["booking_count"], stay_stats["booking_pct"]):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
            f"{val:,}\n({pct:.1%})", ha="center", fontsize=10)

bars2 = ax2.bar(stay_stats["stay_length_group"], stay_stats["cancel_rate"], color=colors)
ax2.set_title("各入住时长组取消率", fontsize=13, fontweight="bold")
ax2.set_xlabel("入住晚数分组")
ax2.set_ylabel("取消率（%）")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
for bar, val in zip(bars2, stay_stats["cancel_rate"]):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.1%}", ha="center", fontsize=11)

fig.tight_layout()
save_figure(fig, "05_stay_length_distribution.png")

**阶段性结论**：
- 绝大多数预订的入住晚数为 2-3 晚，属于典型的短途/周末出行；
- 1 晚的订单取消率最高，可能与其决策门槛低、变更灵活有关；
- 8 晚以上的长住订单占比极小，但 ADR 和预估收入均值最高，属于高价值细分；
- 取消率随着入住晚数增加呈下降趋势，长住客户入住意愿更坚定。

---
## 第 3 步：提前预订时间分布分析

In [ ]:
lt_stats = calculate_lead_time_summary(df)
lt_stats

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = sns.color_palette("Set2", len(lt_stats))

bars1 = ax1.bar(lt_stats["lead_time_group"], lt_stats["booking_count"], color=colors)
ax1.set_title("提前预订时间分布", fontsize=13, fontweight="bold")
ax1.set_xlabel("提前预订天数分组")
ax1.set_ylabel("预订量")
for bar, val, pct in zip(bars1, lt_stats["booking_count"], lt_stats["booking_pct"]):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
            f"{val:,}\n({pct:.1%})", ha="center", fontsize=9)

bars2 = ax2.bar(lt_stats["lead_time_group"], lt_stats["cancel_rate"], color=colors)
ax2.set_title("提前预订时间与取消率关系", fontsize=13, fontweight="bold")
ax2.set_xlabel("提前预订天数分组")
ax2.set_ylabel("取消率（%）")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
for bar, val in zip(bars2, lt_stats["cancel_rate"]):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.1%}", ha="center", fontsize=10)

fig.tight_layout()
save_figure(fig, "05_lead_time_distribution.png")

**阶段性结论**：
- 提前 0-7 天预订的订单占比最高，说明大部分客户是临近入住才下单；
- 提前预订时间与取消率呈强烈的正相关：提前 180 天以上的订单取消率接近 70%，而 0-7 天的仅约 10%；
- 提前预订时间越短，ADR 越高（临期预订价格弹性小，酒店可能维持高价）；
- 180 天以上的远期预订虽然 ADR 较低（可能享受早鸟折扣），但取消风险极大。

---
## 第 4 步：不同 hotel 类型下 lead_time_group 的取消率对比

In [ ]:
cross_lt = df.groupby(["hotel", "lead_time_group"])["is_canceled"].agg(
    booking_count="count", cancel_rate="mean"
).reset_index()
cross_lt["cancel_rate"] = cross_lt["cancel_rate"].round(4)
cross_lt

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
lt_order = ["0-7天", "8-30天", "31-90天", "91-180天", "180天以上"]
hotels = df["hotel"].unique()
colors = sns.color_palette("Set2", len(hotels))

x = np.arange(len(lt_order))
width = 0.35
for i, hotel in enumerate(hotels):
    subset = cross_lt[cross_lt["hotel"] == hotel]
    subset = subset.set_index("lead_time_group").reindex(lt_order)
    ax.bar(x + i * width, subset["cancel_rate"], width,
           color=colors[i], label=hotel)

ax.set_title("不同酒店类型 × 提前预订时间的取消率对比", fontsize=14, fontweight="bold")
ax.set_xlabel("提前预订天数分组")
ax.set_ylabel("取消率（%）")
ax.set_xticks(x + width / 2)
ax.set_xticklabels(lt_order)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend(title="酒店类型")
fig.tight_layout()
save_figure(fig, "05_cancel_rate_by_hotel_lead_time.png")

**阶段性结论**：无论 City Hotel 还是 Resort Hotel，取消率都随提前预订时间增加而上升，趋势高度一致。City Hotel 在各 lead_time 分组中的取消率均高于 Resort Hotel，说明城市酒店的取消风险普遍更高。

---
## 第 5 步：入住时长与取消率的关系（深入分析）

In [ ]:
print("入住时长与取消率（已在第 2 步计算）:")
print(stay_stats[["stay_length_group", "booking_count", "cancel_rate", "avg_adr", "avg_estimated_revenue"]].to_string(index=False))
print(f"\n关键发现：")
print(f"  - 1晚订单取消率最高: {stay_stats[stay_stats['stay_length_group']=='1晚']['cancel_rate'].values[0]:.1%}")
print(f"  - 8晚以上订单取消率最低: {stay_stats[stay_stats['stay_length_group']=='8晚以上']['cancel_rate'].values[0]:.1%}")
print(f"  - 长住订单（8晚+）平均预估收入是 1 晚订单的 {stay_stats[stay_stats['stay_length_group']=='8晚以上']['avg_estimated_revenue'].values[0] / stay_stats[stay_stats['stay_length_group']=='1晚']['avg_estimated_revenue'].values[0]:.0f} 倍")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = sns.color_palette("Set2", len(stay_stats))
bars = ax.bar(stay_stats["stay_length_group"], stay_stats["avg_estimated_revenue"], color=colors)
ax.set_title("各入住时长组的平均每单预估收入", fontsize=14, fontweight="bold")
ax.set_xlabel("入住晚数分组")
ax.set_ylabel("平均预估收入")
for bar, val in zip(bars, stay_stats["avg_estimated_revenue"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            f"{val:,.0f}", ha="center", fontsize=11)
fig.tight_layout()
save_figure(fig, "05_avg_revenue_by_stay_length.png")

**阶段性结论**：长住订单虽然数量少，但单笔预估收入远高于短住订单，且取消率更低，是极具价值的客户群体。1 晚订单取消率高、单笔收入低，可考虑设置最低入住晚数或预付政策以降低风险。

---
## 第 6 步：家庭客户与非家庭客户行为差异

In [ ]:
family_stats = calculate_family_behavior_summary(df)
family_stats

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = [sns.color_palette("Set2")[0], sns.color_palette("Set2")[3]]

ax1, ax2, ax3 = axes

bars1 = ax1.bar(family_stats["family_label"], family_stats["cancel_rate"], color=colors)
ax1.set_title("家庭 vs 非家庭 取消率对比", fontsize=12, fontweight="bold")
ax1.set_ylabel("取消率（%）")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
for bar, val in zip(bars1, family_stats["cancel_rate"]):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.1%}", ha="center", fontsize=11)

bars2 = ax2.bar(family_stats["family_label"], family_stats["avg_total_nights"], color=colors)
ax2.set_title("家庭 vs 非家庭 平均入住晚数", fontsize=12, fontweight="bold")
ax2.set_ylabel("平均入住晚数")
for bar, val in zip(bars2, family_stats["avg_total_nights"]):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f"{val:.1f} 晚", ha="center", fontsize=11)

bars3 = ax3.bar(family_stats["family_label"], family_stats["avg_adr"], color=colors)
ax3.set_title("家庭 vs 非家庭 平均 ADR", fontsize=12, fontweight="bold")
ax3.set_ylabel("平均 ADR")
for bar, val in zip(bars3, family_stats["avg_adr"]):
    ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val:.0f}", ha="center", fontsize=11)

fig.tight_layout()
save_figure(fig, "05_family_behavior.png")

**阶段性结论**：
- 家庭客户仅占约 8-9%，但入住晚数显著长于非家庭客户（平均多 1+ 晚）；
- 家庭客户的取消率明显低于非家庭客户，说明家庭出行决策更慎重、入住意愿更强；
- 家庭客户的 ADR 可能与非家庭客户持平或略低，但考虑到更长的入住晚数，每单预估收入价值更高；
- 家庭客群是值得重点维护的高质量细分市场。

---
## 第 7 步：不同季节的入住行为差异

In [ ]:
season_stats = calculate_season_stay_summary(df)
season_stats

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
colors = sns.color_palette("Set2", len(season_stats))

bars1 = ax1.bar(season_stats["season"], season_stats["avg_total_nights"], color=colors)
ax1.set_title("各季节平均入住晚数", fontsize=13, fontweight="bold")
ax1.set_xlabel("季节")
ax1.set_ylabel("平均入住晚数")
for bar, val in zip(bars1, season_stats["avg_total_nights"]):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{val:.1f} 晚", ha="center", fontsize=11)

bars2 = ax2.bar(season_stats["season"], season_stats["cancel_rate"], color=colors)
ax2.set_title("各季节取消率", fontsize=13, fontweight="bold")
ax2.set_xlabel("季节")
ax2.set_ylabel("取消率（%）")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
for bar, val in zip(bars2, season_stats["cancel_rate"]):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.1%}", ha="center", fontsize=11)

fig.tight_layout()
save_figure(fig, "05_avg_nights_by_season.png")

**阶段性结论**：
- 夏季平均入住晚数最长，与度假旺季吻合，且夏季 ADR 和预估收入均为全年最高；
- 冬季虽然预订量低，但平均入住晚数可能不低（部分冬季长住游客）；
- 夏季取消率相对较高（旺季选择多，客户可能多家比价后取消），需注意旺季的取消管理；
- 春季和秋季是平季，入住行为较为均衡。

---
## 第 8 步：不同 hotel 类型下的入住行为差异

In [ ]:
hotel_stats = calculate_hotel_stay_summary(df)
hotel_stats

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
colors = sns.color_palette("Set2", len(hotel_stats))

bars1 = ax1.bar(hotel_stats["hotel"], hotel_stats["avg_total_nights"], color=colors)
ax1.set_title("各酒店类型平均入住晚数", fontsize=13, fontweight="bold")
ax1.set_xlabel("酒店类型")
ax1.set_ylabel("平均入住晚数")
for bar, val in zip(bars1, hotel_stats["avg_total_nights"]):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f"{val:.1f} 晚", ha="center", fontsize=11)

bars2 = ax2.bar(hotel_stats["hotel"], hotel_stats["avg_lead_time"], color=colors)
ax2.set_title("各酒店类型平均提前预订天数", fontsize=13, fontweight="bold")
ax2.set_xlabel("酒店类型")
ax2.set_ylabel("平均提前预订天数")
for bar, val in zip(bars2, hotel_stats["avg_lead_time"]):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{val:.0f} 天", ha="center", fontsize=11)

fig.tight_layout()
save_figure(fig, "05_avg_lead_time_by_hotel.png")

In [ ]:
cross_stay_hotel = df.groupby(["hotel", "stay_length_group"]).size().unstack(fill_value=0)
cross_stay_pct = cross_stay_hotel.div(cross_stay_hotel.sum(axis=1), axis=0).round(3)
print("各酒店类型 × 入住时长分组 占比:")
cross_stay_pct

**阶段性结论**：
- Resort Hotel 的平均入住晚数显著长于 City Hotel，符合度假酒店定位（客人倾向于停留更长时间）；
- City Hotel 的平均提前预订天数更长，可能与商务出行提前规划较多有关；
- Resort Hotel 的 4-7 晚和 8 晚以上占比更高，而 City Hotel 的 1 晚占比更高（商务短差）；
- 取消率方面 City Hotel 高于 Resort Hotel，需结合城市酒店客户流动性大的特点制定差异化策略。

---
## 综合结论

本 Notebook 从入住时长与提前预订行为维度进行了深入分析，核心发现如下：

1. **入住时长集中**：2-3 晚的短途出行是最主要的入住模式，占比超过 50%。1 晚订单取消率最高、单笔收入最低，长住订单（8 晚以上）则兼具低取消率与高客单价。

2. **提前预订与取消强相关**：提前天数越长，取消率越高——180 天以上订单取消率接近 70%，而 0-7 天仅约 10%。远期预订是取消风险的核心来源。

3. **临期预订 ADR 更高**：临近入住才预订的客户往往接受更高房价，而早鸟预订虽然锁定入住率但 ADR 较低且取消风险大，需要在早鸟折扣与收益之间权衡。

4. **家庭客户质量高**：家庭客户入住晚数更长、取消率更低，是值得重点维护的高质量客群，可针对家庭客户推出亲子套餐、连住优惠等产品。

5. **季节行为差异显著**：夏季入住晚数最长、ADR 和收入最高，但取消率也较高（旺季客户选择多）；冬季预订量虽少但入住行为相对稳定。

6. **酒店类型行为分化**：Resort Hotel 客户入住更久、取消率更低，是典型的度假行为；City Hotel 客户以短住为主、取消率更高，需针对城市酒店特点设计不同的定价和取消政策。

7. **运营建议**：
   - 对远期预订（90 天以上）引入适度预付或非退款政策以控制取消风险；
   - 对 1 晚订单评估最低入住晚数策略以提升单笔收入；
   - 加大针对家庭客户的营销投入，提升复购率；
   - 旺季加强收益管理，淡季通过促销套餐平衡入住率。